# 02 - Synthetic fixture and donor split

This notebook is updated by more than one commit. Each commit adds a dated section.

| Section | Commit | Date |
|---|---|---|
| C03 synthetic fixture | `test: add deterministic synthetic multimodal fixtures` | 2026-07-27 |
| C04 donor-held-out split | `feat: add donor-held-out split validation` | 2026-07-27 |

## C03 - Deterministic synthetic fixture (2026-07-27)

- **Meeting item:** Prototype validation
- **Commit:** C03 - `test: add deterministic synthetic multimodal fixtures`
- **Commit achievement:** creates a deterministic two-view test world with donor nuisance.
- **Data mode:** `synthetic`. Every array below is generated in memory; nothing is read from disk.
- **Evidence labels used:** verified, unknown
- **Human gate:** does the same seed reproduce identical arrays while a different seed changes them?
- **Stop condition:** NaN or infinity, missing identifiers, biological names, or nondeterminism.

All generation logic lives in `src/p22/testing/synthetic.py`.

In [ ]:
import numpy as np
import pandas as pd

import p22
from p22.testing import make_synthetic_multimodal

dataset = make_synthetic_multimodal(seed=42)

print(f"data mode: {dataset.generation['data_mode']}")
print(f"generator: {dataset.generation['generator']}")
print(f"view_a: {dataset.view_a.shape} {dataset.view_a.dtype}")
print(f"view_b: {dataset.view_b.shape} {dataset.view_b.dtype}")
print(f"cells: {dataset.n_cells}, donors: {dataset.n_donors}, classes: {dataset.n_classes}")
print(f"finite view_a: {bool(np.isfinite(dataset.view_a).all())}")
print(f"finite view_b: {bool(np.isfinite(dataset.view_b).all())}")
dataset.metadata.head()

### Label mix and donor structure (evidence label: verified)

Class balance is reported, never promised. Donors carry their own class mix, which is why
donor-blind evaluation would flatter a model.

In [ ]:
print("cells per class:")
print(dataset.label_distribution().to_string())
print("")
print("donor by class cell counts:")
print(dataset.donor_label_counts().to_string())

In [ ]:
donor_summary = (
    dataset.metadata.groupby("donor_id")[["view_a_total", "view_b_total"]]
    .agg(["mean", "std"])
    .round(3)
)
spread_between = dataset.metadata.groupby("donor_id")["view_b_total"].mean().std()
spread_within = dataset.metadata.groupby("donor_id")["view_b_total"].std().mean()
print(f"view_b donor-mean spread between donors: {spread_between:.3f}")
print(f"view_b mean spread within a donor: {spread_within:.3f}")
print("donor shift is present by construction; view B carries a direct donor offset")
donor_summary

### Determinism (evidence label: verified)

In [ ]:
repeat_same = make_synthetic_multimodal(seed=42)
other_seed = make_synthetic_multimodal(seed=43)

checks = {
    "same seed, view_a identical": np.array_equal(dataset.view_a, repeat_same.view_a),
    "same seed, view_b identical": np.array_equal(dataset.view_b, repeat_same.view_b),
    "same seed, labels identical": np.array_equal(dataset.labels, repeat_same.labels),
    "different seed, view_a changes": not np.array_equal(dataset.view_a, other_seed.view_a),
    "different seed, labels change": not np.array_equal(dataset.labels, other_seed.labels),
    "donor identifiers stable across seeds": np.array_equal(
        dataset.donor_ids, other_seed.donor_ids
    ),
}
print(pd.Series(checks).to_string())
assert all(checks.values()), checks

### Why the signal is imperfect (evidence label: verified)

Each view reads a different half of the latent space strongly and the other half weakly, so
neither view alone contains the whole label signal. Model fitting is not done here; that
starts in C07 and C10.

In [ ]:
weights = dataset.generation["view_latent_weights"]
print(f"latent weights, view_a: {weights['view_a']}")
print(f"latent weights, view_b: {weights['view_b']}")
print(f"class signal: {dataset.generation['class_signal']}")
print(f"donor nuisance: {dataset.generation['donor_nuisance']}")
print(f"latent noise: {dataset.generation['noise']}")
print("")
per_class = (
    pd.DataFrame(dataset.view_a[:, :4], columns=[f"view_a_f{i}" for i in range(4)])
    .assign(label_name=dataset.metadata["label_name"].to_numpy())
    .groupby("label_name")
    .mean()
    .round(3)
)
print("per-class feature means overlap, so classes are not separable by inspection:")
per_class

In [ ]:
import re

assert dataset.generation["data_mode"] == p22.SYNTHETIC
assert np.isfinite(dataset.view_a).all() and np.isfinite(dataset.view_b).all()
assert dataset.metadata.notna().all().all()
assert len(set(dataset.cell_ids)) == dataset.n_cells
assert set(np.unique(dataset.labels)) == set(range(dataset.n_classes))
assert all(re.fullmatch(r"donor_\d+", value) for value in dataset.donor_ids)
assert all(re.fullmatch(r"cell_\d+", value) for value in dataset.cell_ids)
print("all C03 gate checks passed")

### Unknown (evidence label: unknown)

- Whether this synthetic world resembles any real dataset. It is a controlled test world, not a simulation of one.
- What accuracy is reachable here; that is measured in C10 with donor-held-out splits and five seeds.

## C04 - Donor-held-out split (2026-07-27)

- **Meeting item:** Donor-aware evaluation
- **Commit:** C04 - `feat: add donor-held-out split validation`
- **Commit achievement:** proves the split happens by donor, not by cell.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified, unknown
- **Human gate:** does the notebook show zero donor overlap and complete cell assignment?
- **Stop condition:** any donor appears in more than one split.

Splitting logic lives in `src/p22/data/splits.py`. `make_donor_split` validates its own
output before returning, so an invalid split cannot leave the function.

In [ ]:
from p22.data import donor_split_problems, make_donor_split, validate_donor_split

split = make_donor_split(
    dataset.donor_ids,
    labels=dataset.labels,
    val_fraction=0.25,
    test_fraction=0.25,
    seed=0,
)

print(f"split unit: {split.summary['split_unit']}")
print(f"train donors: {split.train_donors}")
print(f"val donors:   {split.val_donors}")
print(f"test donors:  {split.test_donors}")
print("")
split.as_frame()

In [ ]:
report = validate_donor_split(split, dataset.donor_ids, labels=dataset.labels)
assigned = np.concatenate([split.index(name) for name in ("train", "val", "test")])

print(f"donor overlap between splits: {report['donor_overlap']}")
print(f"unassigned cells: {report['unassigned_cells']}")
print(f"cells assigned once: {np.unique(assigned).size} of {dataset.n_cells}")
print(f"requested fractions: {report['requested_fractions']}")
print(f"realized donor counts: {report['realized_donor_counts']}")
print(f"realized cell counts: {report['realized_cell_counts']}")
print("")
print("observed label counts per split (reported, not enforced):")
print(pd.DataFrame(report["label_distribution"]).fillna(0).astype(int).to_string())

### Why donor splitting is not optional (evidence label: verified)

The cell below shuffles cells instead of donors, purely to show the failure mode. Its output
is never used downstream.

In [ ]:
rng = np.random.default_rng(0)
shuffled = rng.permutation(dataset.n_cells)
cut = int(0.75 * dataset.n_cells)
cell_level_train = set(dataset.donor_ids[shuffled[:cut]])
cell_level_test = set(dataset.donor_ids[shuffled[cut:]])

print(f"cell-level split: donors shared between train and test: {len(cell_level_train & cell_level_test)} of {dataset.n_donors}")
print(f"donor-level split: donors shared between train and test: {len(set(split.train_donors) & set(split.test_donors))} of {dataset.n_donors}")
print("")
print("a cell-level split lets every donor appear on both sides, so the test estimate")
print("measures memorised donor structure as well as label signal")

In [ ]:
import dataclasses

leaking = dataclasses.replace(split, val_donors=split.val_donors + (split.train_donors[0],))
print("rejection of a leaking split:")
for problem in donor_split_problems(leaking, dataset.donor_ids):
    print(f"  - {problem}")

incomplete = dataclasses.replace(split, test_index=split.test_index[:-1])
print("rejection of an incomplete assignment:")
for problem in donor_split_problems(incomplete, dataset.donor_ids):
    print(f"  - {problem}")

for bad in ({"val_fraction": 0.0}, {"val_fraction": 0.6, "test_fraction": 0.5}):
    try:
        make_donor_split(dataset.donor_ids, **bad)
    except ValueError as error:
        print(f"rejected {bad}: {error}")

try:
    make_donor_split(["donor_00", None, "donor_01", "donor_02"])
except ValueError as error:
    print(f"rejected null identifier: {error}")

In [ ]:
same_seed = make_donor_split(dataset.donor_ids, val_fraction=0.25, test_fraction=0.25, seed=0)
other_seed = make_donor_split(dataset.donor_ids, val_fraction=0.25, test_fraction=0.25, seed=1)

assert set(split.train_donors) & set(split.val_donors) == set()
assert set(split.train_donors) & set(split.test_donors) == set()
assert set(split.val_donors) & set(split.test_donors) == set()
assert np.array_equal(np.unique(assigned), np.arange(dataset.n_cells))
assert same_seed.train_donors == split.train_donors
assert other_seed.train_donors != split.train_donors
for name in ("train", "val", "test"):
    assert set(dataset.donor_ids[split.index(name)]) == set(split.donors(name))
print("all C04 gate checks passed")

### Unknown (evidence label: unknown)

- Whether the label mix per split is close enough for stable comparison; it is reported per split, not enforced.
- Whether donor identifiers will exist in an approved dataset. If they do not, the sequence stops rather than falling back to cell-level splitting.

## Gate

**PASS**

- **Observed (C03):** seed 42 produces a 240-cell, 12-donor, 3-class world with `view_a` of shape (240, 16) and `view_b` of shape (240, 12), no NaN or infinity, unique neutral identifiers, uneven per-donor class mix, a donor offset in view B, identical arrays on a repeated seed, and different arrays on a different seed.
- **Observed (C04):** donor split of 6 train, 3 validation, and 3 test donors with zero donor overlap, all 240 cells assigned exactly once, label counts reported per split, identical split on a repeated seed and a different split on a different seed, and explicit rejection of donor overlap, incomplete assignment, invalid fractions, and null identifiers. The contrast cell shows a cell-level split shares 11 of 12 donors between train and test.
- **Remains unknown:** reachable accuracy in this world, whether per-split label mix is close enough for stable comparison, and whether an approved dataset will carry donor identifiers.
- **Next decision owner:** researcher continues with C05 (train-only preprocessing).